# Baseline Evaluation
Evaluate `bhadresh-savani/bert-base-go-emotion` on the GoEmotions validation set.
Records accuracy, macro F1, weighted F1, and per-class precision / recall / F1.

In [1]:
import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


## 1. Label names

In [2]:
LABEL_NAMES = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]
NUM_LABELS = len(LABEL_NAMES)
print(f'{NUM_LABELS} classes')

28 classes


## 2. Load baseline model
`bhadresh-savani/bert-base-go-emotion` is already fine-tuned on GoEmotions — we evaluate it as-is (no further training).

In [3]:
MODEL_NAME = 'bhadresh-savani/bert-base-go-emotion'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

# Verify label order matches our dataset
model_labels = [model.config.id2label[i] for i in range(NUM_LABELS)]
assert model_labels == LABEL_NAMES, (
    f'Label mismatch!\nmodel: {model_labels}\nours:  {LABEL_NAMES}'
)
print('Label order verified — matches dataset')

config.json: 0.00B [00:00, ?B/s]

d:\anaconda3\envs\emotion_seq\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gymyo\.cache\huggingface\hub\models--bhadresh-savani--bert-base-go-emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bhadresh-savani/bert-base-go-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Label order verified — matches dataset


## 3. Load validation split from disk

In [ ]:
tokenized = load_from_disk('processed/go_emotions_tokenized')
val_split = tokenized['validation']
val_split.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])

val_loader = DataLoader(val_split, batch_size=32, shuffle=False)
print(f'Validation examples: {len(val_split)}  |  Batches: {len(val_loader)}')

## 4. Run inference

In [ ]:
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['label']

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

        if (batch_idx + 1) % 50 == 0:
            print(f'  batch {batch_idx + 1}/{len(val_loader)}')

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print(f'Done. Total predictions: {len(all_preds)}')

## 5. Compute overall metrics

In [ ]:
accuracy    = accuracy_score(all_labels, all_preds)
macro_f1    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print('=== Baseline Overall Metrics ===')
print(f'  Accuracy    : {accuracy:.4f}')
print(f'  Macro F1    : {macro_f1:.4f}')
print(f'  Weighted F1 : {weighted_f1:.4f}')

## 6. Per-class report

In [ ]:
report = classification_report(
    all_labels, all_preds,
    target_names=LABEL_NAMES,
    zero_division=0,
    output_dict=True
)

per_class = pd.DataFrame(report).T.loc[LABEL_NAMES, ['precision', 'recall', 'f1-score', 'support']]
per_class = per_class.astype({'support': int})
per_class = per_class.sort_values('f1-score', ascending=False)

pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('=== Baseline Per-Class Metrics (sorted by F1) ===')
display(per_class)

## 7. Save results to CSV

In [ ]:
import os, json

os.makedirs('results/summary', exist_ok=True)

# Per-class table
per_class.to_csv('results/summary/baseline_per_class.csv')

# Overall summary
summary = {
    'model':        MODEL_NAME,
    'split':        'validation',
    'n_samples':    int(len(all_labels)),
    'accuracy':     round(accuracy, 6),
    'macro_f1':     round(macro_f1, 6),
    'weighted_f1':  round(weighted_f1, 6),
}
with open('results/summary/baseline_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved:')
print('  results/summary/baseline_summary.json')
print('  results/baseline_per_class.csv')